# CTD Profile Grapher

Interactive depth profiles from Sea-Bird `.cnv` files — no Excel step, no Google Drive.

**How to use**
1. Run **Setup** once per session.
2. Run **1 · Load** and pick your `.cnv` files with the upload button.
3. Run **2 · Plot**. To focus on part of the water column, type a depth into the `bottom_m` box on the right and run that cell again — the graphs redraw and the x axis rescales to the values inside that window. Your files stay loaded, so there is no need to upload again.

**Naming:** the filename becomes the legend label, with underscores turned into spaces — `Station_1.cnv` → **Station 1**, `East_Passage.cnv` → **East Passage**. Stations sort naturally (1, 2, … 10, 11).

**Outputs** land in `/content/CTD_output/`:
- `CTD_profiles.html` — one self-contained interactive file, works offline, safe to open or embed anywhere
- `png/*.png` — high-resolution static copies for slides and print

The sensor set is read from each file's own header, so UW-Tacoma (transmissivity) and Friday Harbor (turbidity + pH) files both work with no setting to change.

In [ ]:
#@title Setup — run once per session
!pip install -q "kaleido==0.2.1" 2>/dev/null

import os, re, glob
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# ─────────────── settings ───────────────
CNV_FOLDER   = "/content/CTD Files"  # None → always use the upload button
LINE_SHAPE   = "spline"              # "spline" (smooth) or "linear" (raw bins)
SHOW_MARKERS = False                 # True → dot at each 1-db bin
LINE_WIDTH   = 1.5
X_PAD_FRAC   = 0.05                  # breathing room at left/right edges (5%)
Y_PAD_FRAC   = 0.02
EXPORT_PNG   = True
OUT_DIR      = "/content/CTD_output"
# ────────────────────────────────────────

# Canonical variable → Sea-Bird short names to look for, in priority order.
# Whatever a file actually contains is what gets plotted.
VARIABLES = [
    ("Temperature",       "°C",    ["tv290c", "t090c", "tv190c", "t190c", "t090"]),
    ("Salinity",          "PSU",        ["sal00", "sal11"]),
    ("Density (sigma-t)", "kg/m³", ["sigma-t00", "sigma-t11"]),
    ("Dissolved Oxygen",  "mL/L",       ["sbeox0ml/l", "sbeox1ml/l"]),
    ("Fluorescence",      "mg/m³", ["fleco-afl", "flecoafl", "flsp", "flc"]),
    ("Beam Transmission", "%",          ["cstartr0", "cstartr1", "xmiss"]),
    ("Turbidity",         "NTU",        ["turbwetntu0", "turbwetntu1", "obs"]),
    ("pH",                "",           ["ph"]),
]
DEPTH_CANDIDATES = ["depsm", "depfm", "prdm", "pr"]

PALETTE = ["#1f77b4", "#d62728", "#2ca02c", "#ff7f0e", "#9467bd",
           "#8c564b", "#e377c2", "#17becf", "#bcbd22", "#7f7f7f"]


def natkey(s):
    """Sort so Station 2 comes before Station 10."""
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r"(\d+)", str(s))]


def station_label(filename):
    """Station_1.cnv -> 'Station 1';  East_Passage.cnv -> 'East Passage'"""
    return os.path.splitext(os.path.basename(filename))[0].replace("_", " ").strip()


def find_col(df, candidates):
    """First matching column, case-insensitive; first occurrence wins on duplicates."""
    lower = {}
    for c in df.columns:
        lower.setdefault(str(c).lower(), c)
    for cand in candidates:
        if cand in lower:
            return lower[cand]
    return None


def parse_cnv(text, source_name):
    """Parse a Sea-Bird .cnv. Columns come from the '# name' header lines and the
    data starts after *END*, so header length never has to be hardcoded."""
    lines = text.splitlines()
    col_names, bad_flag, end_idx, processing = {}, -9.99e-29, None, set()

    for i, raw in enumerate(lines):
        s = raw.strip()
        if s.upper() == "*END*":
            end_idx = i
            break
        m = re.match(r"#\s*name\s+(\d+)\s*=\s*([^:]+):", s)
        if m:
            col_names[int(m.group(1))] = m.group(2).strip()
            continue
        m = re.match(r"#\s*bad_flag\s*=\s*(\S+)", s)
        if m:
            try:
                bad_flag = float(m.group(1))
            except ValueError:
                pass
            continue
        low = s.lower()
        for tag in ("loopedit", "binavg", "wfilter", "filter", "derive"):
            if low.startswith("# " + tag):
                processing.add(tag)

    if end_idx is None:
        raise ValueError(f"{source_name}: no *END* marker — is this a Sea-Bird .cnv?")
    if not col_names:
        raise ValueError(f"{source_name}: no '# name' column definitions in header.")

    ordered = [col_names[k] for k in sorted(col_names)]
    rows = []
    for raw in lines[end_idx + 1:]:
        parts = raw.split()
        if len(parts) != len(ordered):
            continue
        try:
            rows.append([float(x) for x in parts])
        except ValueError:
            continue

    df = pd.DataFrame(rows, columns=ordered)
    if not df.empty:
        # bad_flag is ~1e-29, so the comparison must be purely relative (atol=0),
        # otherwise every near-zero reading would be wiped out.
        mask = np.isclose(df.values.astype(float), bad_flag, rtol=1e-6, atol=0.0)
        df = df.mask(pd.DataFrame(mask, index=df.index, columns=df.columns))
    return df, processing


def load_files():
    """Read .cnv from CNV_FOLDER if it has any, else show the upload button."""
    if CNV_FOLDER and os.path.isdir(CNV_FOLDER):
        paths = sorted(
            [p for p in glob.glob(os.path.join(CNV_FOLDER, "*")) if p.lower().endswith(".cnv")],
            key=natkey,
        )
        if paths:
            print(f"Reading {len(paths)} file(s) from {CNV_FOLDER}\n")
            # .cnv headers are cp1252, not UTF-8 (e.g. the theta in sigma-theta)
            return {os.path.basename(p): open(p, "r", encoding="latin-1").read() for p in paths}

    from google.colab import files as colab_files
    print("Select one or more .cnv files:")
    return {n: b.decode("latin-1") for n, b in colab_files.upload().items()}


def _padded(lo, hi, frac):
    """Range with breathing room. Falls back sensibly if the series is flat."""
    span = hi - lo
    pad = span * frac if span > 0 else (abs(hi) * frac if hi else 1.0) or 1.0
    return lo - pad, hi + pad


def build_figures(stations, depth_min=None, depth_max=None):
    """stations: list of (label, dataframe). One figure per detected variable,
    every station overlaid, colours consistent across all figures.

    depth_min/depth_max crop the water column; the x axis then rescales to the
    values actually inside that window, which is the point of zooming in."""
    figs = []
    for label_txt, unit, cands in VARIABLES:
        traces = []
        xlo = ylo = np.inf
        xhi = yhi = -np.inf
        for i, (st, df) in enumerate(stations):
            xcol, ycol = find_col(df, cands), find_col(df, DEPTH_CANDIDATES)
            if xcol is None or ycol is None:
                continue
            sub = df[[xcol, ycol]].dropna()
            if depth_min is not None:
                sub = sub[sub[ycol] >= depth_min]
            if depth_max is not None:
                sub = sub[sub[ycol] <= depth_max]
            if sub.empty:
                continue
            xlo, xhi = min(xlo, sub[xcol].min()), max(xhi, sub[xcol].max())
            ylo, yhi = min(ylo, sub[ycol].min()), max(yhi, sub[ycol].max())
            traces.append(go.Scatter(
                x=sub[xcol], y=sub[ycol], name=st,
                mode="lines+markers" if SHOW_MARKERS else "lines",
                line=dict(shape=LINE_SHAPE, width=LINE_WIDTH,
                          color=PALETTE[i % len(PALETTE)]),
                marker=dict(size=4),
                hovertemplate=(f"<b>{st}</b><br>{label_txt}: %{{x:.3f}} {unit}"
                               f"<br>Depth: %{{y:.2f}} m<extra></extra>"),
            ))
        if not traces:
            continue

        x0, x1 = _padded(xlo, xhi, X_PAD_FRAC)
        y0, y1 = _padded(ylo, yhi, Y_PAD_FRAC)
        # When the window reaches the surface, keep 0 m a little INSIDE the axis.
        # Pinning the range to exactly 0.0 puts the tick on the frame itself, where
        # it does not draw until the plot is panned.
        at_surface = depth_min is None or depth_min <= 0
        if at_surface:
            y0 = -(yhi - ylo) * Y_PAD_FRAC or -1.0
        axis = f"{label_txt} [{unit}]" if unit else label_txt

        fig = go.Figure(traces)
        fig.update_layout(
            title=axis, xaxis_title=axis, yaxis_title="Depth [m]",
            xaxis=dict(range=[x0, x1]),
            yaxis=dict(range=[y1, y0]),          # descending → depth increases downward
            template="plotly_white", hovermode="closest",
            width=760, height=620,
            legend=dict(title="Station"), margin=dict(l=70, r=30, t=60, b=60),
        )
        figs.append((label_txt, fig))
    return figs


def write_combined_html(figs, path, title="CTD Profiles"):
    """All figures in ONE self-contained file. plotly.js is inlined once, so the
    file is ~4 MB but needs no internet — safe for a presentation."""
    parts = [f.to_html(full_html=False,
                       include_plotlyjs="inline" if i == 0 else False)
             for i, (_, f) in enumerate(figs)]
    html = (
        '<!doctype html><html><head><meta charset="utf-8">'
        f"<title>{title}</title><style>"
        "body{font-family:system-ui,-apple-system,'Segoe UI',sans-serif;margin:24px;"
        "background:#fff;color:#111}h1{font-size:20px;font-weight:600}"
        ".grid{display:flex;flex-wrap:wrap;gap:16px}</style></head><body>"
        f"<h1>{title}</h1><div class=\"grid\">{''.join(parts)}</div></body></html>"
    )
    with open(path, "w", encoding="utf-8") as f:
        f.write(html)


def export_figures(figs, tag="", want_png=True):
    """Write the interactive HTML (+ optional PNGs) and return a status string."""
    os.makedirs(os.path.join(OUT_DIR, "png"), exist_ok=True)
    html_path = os.path.join(OUT_DIR, "CTD_profiles.html")
    write_combined_html(figs, html_path, title=f"CTD Profiles{tag}")
    msg = f"{html_path} ({os.path.getsize(html_path)/1e6:.1f} MB)"
    if want_png and EXPORT_PNG:
        try:
            for name, fig in figs:
                safe = re.sub(r"[^A-Za-z0-9]+", "_", name).strip("_")
                fig.write_image(os.path.join(OUT_DIR, "png", f"{safe}.png"), scale=3)
            msg += f" · {len(figs)} PNGs"
        except Exception as e:
            msg += f" · PNG skipped ({type(e).__name__})"
    return msg


print("Setup complete.  markers:", SHOW_MARKERS, "· x-padding:", f"{X_PAD_FRAC:.0%}")

In [ ]:
#@title 1 · Load — upload your .cnv files
raw = load_files()

stations = []
for fn in sorted(raw, key=natkey):
    try:
        df, proc = parse_cnv(raw[fn], fn)
    except ValueError as e:
        print(f"  SKIPPED  {e}")
        continue
    if df.empty:
        print(f"  SKIPPED  {fn}: no data rows after *END*")
        continue

    label = station_label(fn)
    stations.append((label, df))
    dcol = find_col(df, DEPTH_CANDIDATES)
    drange = f"{df[dcol].min():.1f}–{df[dcol].max():.1f} m" if dcol else "no depth column"
    print(f"  {label:<28} {len(df):>4} rows   {drange:<16} "
          f"processing: {', '.join(sorted(proc)) or 'none'}")
    if "loopedit" not in proc:
        print(f"  NOTE  {label}: no loopedit in header — this file may still contain "
              f"the upcast, so the profile can double back on itself.")

if not stations:
    raise SystemExit("No readable .cnv files found.")

DEEPEST = max(df[find_col(df, DEPTH_CANDIDATES)].max() for _, df in stations)
print(f"\n{len(stations)} station(s) loaded · deepest reading {DEEPEST:.1f} m")
print("Now run the Plot cell.")

In [ ]:
#@title 2 · Plot — set the depth window, then run
#@markdown Leave both blank for the whole cast. To focus on the top 20 m, put `20` in **bottom_m** and run again.
top_m    = "" #@param {type:"string"}
bottom_m = "" #@param {type:"string"}

try:
    stations
except NameError:
    raise SystemExit("Run the Load cell first.")


def _num(s):
    s = str(s).strip()
    if not s:
        return None
    try:
        return float(s)
    except ValueError:
        print(f"  ignoring '{s}' — not a number")
        return None


dmin, dmax = _num(top_m), _num(bottom_m)
if dmin is not None and dmax is not None and dmin > dmax:
    dmin, dmax = dmax, dmin

lo = f"{dmin:g}" if dmin is not None else "0"
hi = f"{dmax:g}" if dmax is not None else f"{DEEPEST:.0f}"
figs = build_figures(stations, dmin, dmax)

if not figs:
    print(f"No data between {lo} and {hi} m. Deepest reading in this set is {DEEPEST:.1f} m.")
else:
    print(f"Showing {lo}–{hi} m · {len(figs)} graphs · "
          + ", ".join(n for n, _ in figs))
    print("Saved:", export_figures(figs, tag=f" · {lo}–{hi} m"))
    # Plain fig.show() with no clear_output(). Colab's plotly renderer attaches a
    # MutationObserver that purges the plot when its output element is rebuilt, so
    # clearing and redrawing the cell destroys the figures as they arrive.
    for _, f in figs:
        f.show()